In [ ]:
# pip install transformers torch requests beautifulsoup4 tldextract --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import tldextract

class SourceTrust:
    """
    Replacement class for the broken sourcetrust package.
    This version includes scoring and works out of the box.
    """

    WIKI_API = "https://en.wikipedia.org/api/rest_v1/page/summary/{}"

    CATEGORY_MAP = {
        "fake news": "fake",
        "conspiracy theory": "problematic",
        "pseudoscience": "problematic",
        "propaganda": "problematic",
        "satire": "neutral",
        "newspaper": "trustworthy",
        "news website": "trustworthy",
        "news agency": "trustworthy",
    }

    def normalize_domain(self, domain):
        ext = tldextract.extract(domain)
        return f"{ext.domain}.{ext.suffix}"

    def fetch_wikipedia_summary(self, name):
        try:
            url = self.WIKI_API.format(name)
            r = requests.get(url, timeout=6)
            if r.status_code == 200:
                return r.json().get("extract", "").lower()
        except:
            pass
        return ""

    def classify_from_wikipedia(self, domain):
        query = domain.replace(".", "_")
        text = self.fetch_wikipedia_summary(query)

        if not text:
            return "unknown"

        for kw, cat in self.CATEGORY_MAP.items():
            if kw in text:
                return cat

        return "neutral"

    def check_domain(self, domain):
        domain = self.normalize_domain(domain)
        category = self.classify_from_wikipedia(domain)

        score_map = {
            "trustworthy": 0.9,
            "neutral": 0.6,
            "unknown": 0.5,
            "problematic": 0.2,
            "fake": 0.1
        }

        return {
            "domain": domain,
            "category": category,
            "score": score_map.get(category, 0.5)
        }


In [3]:
from bs4 import BeautifulSoup

def extract_article_text(url: str):
    """Extract readable article text from a webpage."""
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove JS, styles
    for tag in soup(["script", "style", "noscript"]):
        tag.extract()

    # Extract visible paragraphs
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    return "\n".join(paragraphs)


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "mrm8488/bert-tiny-finetuned-fake-news-detection"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

label_map = {0: "real", 1: "fake"}

def article_credibility(text: str):
    """Score the article using a pretrained fake-news classifier."""
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()

    real_prob = float(probs[0])
    fake_prob = float(probs[1])
    label = label_map[int(torch.argmax(probs))]

    return {
        "real_probability": real_prob,
        "fake_probability": fake_prob,
        "label": label,
        "trust_score": real_prob
    }


/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def combined_reputation(url: str):
    print("Extracting article text...")
    article_text = extract_article_text(url)

    print("Scoring article credibility...")
    article_score = article_credibility(article_text)

    print("Scoring domain reputation...")
    st = SourceTrust()
    domain_score = st.check_domain(url)

    # weighted blend: 60% article, 40% domain
    combined = round(
        0.6 * article_score["trust_score"] +
        0.4 * domain_score["score"],
        3
    )

    return {
        "url": url,
        "article_score": article_score,
        "domain_score": domain_score,
        "combined_reputation": combined
    }


In [10]:
trustworthy = "https://www.nytimes.com/2025/11/19/us/politics/comey-vindictive-prosecution-trump.html"

result = combined_reputation(trustworthy)
result


Extracting article text...
Scoring article credibility...
Scoring domain reputation...


{'url': 'https://www.nytimes.com/2025/11/19/us/politics/comey-vindictive-prosecution-trump.html',
 'article_score': {'real_probability': 0.0010321179870516062,
  'fake_probability': 0.9989678859710693,
  'label': 'fake',
  'trust_score': 0.0010321179870516062},
 'domain_score': {'domain': 'nytimes.com',
  'category': 'unknown',
  'score': 0.5},
 'combined_reputation': 0.201}

In [11]:
untrustworthy = "https://www.infowars.com"

result = combined_reputation(untrustworthy)
result


Extracting article text...
Scoring article credibility...
Scoring domain reputation...


{'url': 'https://www.infowars.com',
 'article_score': {'real_probability': 0.0008623414323665202,
  'fake_probability': 0.9991376399993896,
  'label': 'fake',
  'trust_score': 0.0008623414323665202},
 'domain_score': {'domain': 'infowars.com',
  'category': 'unknown',
  'score': 0.5},
 'combined_reputation': 0.201}